# 03 - CoNLL-2003 Baseline Fine-tune

Fine-tunes `distilbert-base-cased` for token classification on the CoNLL-2003 train split.
This checkpoint is the shared starting point for the whole comparison:

- arm 1 (`05_fewshot_finetune_baseline.ipynb`) fine-tunes it directly on k target examples
- arm 2 (`04` + `06`) continues MLM pretraining on target text first, then fine-tunes
- this notebook also measures the baseline's cross-domain "degradation" on WNUT-17 and
  SciERC (see the Decision 2 markdown below for how we deal with the label-set mismatch)

Outputs:

- `models/baseline_conll/` -- fine-tuned checkpoint + tokenizer
- `results/baseline_results.json` -- in-domain and cross-domain metrics
- `utils/ner_common_utils.py` -- the shared helper module every later notebook imports
- `data/processed/eval_subsets/` -- the fixed test subsamples reused by every method


In [ ]:
%pip install -q transformers datasets accelerate seqeval

## Shared utility module


In [ ]:
%%writefile ner_common_utils.py

import json
import random
from pathlib import Path

import numpy as np

BUDGETS = [50, 100, 200]
SEEDS = [13, 42, 101]
TARGET_DATASETS = ['wnut17', 'scierc']

EVAL_SUBSET_SIZE = 200
EVAL_SUBSET_SEED = 42


def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def save_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')


def mount_drive_if_colab():
    if Path('/content').exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        except Exception:
            pass


def resolve_processed_dir():

    candidates = [
        Path('/content/drive/MyDrive/AAI590/data/processed'),
        Path('/content/drive/MyDrive/ner_capstone/data/processed'),
        Path('/content/ner_capstone/data/processed'),
        Path.cwd() / 'data' / 'processed',
        Path.cwd().parent / 'data' / 'processed',
    ]
    processed_dir = next((p for p in candidates if p.exists()), None)
    if processed_dir is None:
        raise FileNotFoundError(
            'Could not find data/processed. Run 00_collect_datasets_colab.ipynb first '
            '(or put the data folder in Drive / the local repo), then rerun.'
        )
    return processed_dir


def resolve_output_root(processed_dir):

    return Path(processed_dir).parent.parent


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass 


def pick_device():
    import torch
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'  
    return 'cpu'


def compute_entity_f1(true_tags, pred_tags):
    from seqeval.metrics import classification_report
    report = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    micro = report['micro avg']
    per_type = {}
    for name, vals in report.items():
        if name in ('micro avg', 'macro avg', 'weighted avg'):
            continue
        per_type[name] = {
            'precision': float(vals['precision']),
            'recall': float(vals['recall']),
            'f1': float(vals['f1-score']),
            'support': int(vals['support']),
        }
    return {
        'precision': float(micro['precision']),
        'recall': float(micro['recall']),
        'f1': float(micro['f1-score']),
        'support': int(micro['support']),
        'per_type': per_type,
    }


def make_or_load_eval_subset(processed_dir, dataset_name,
                             n=EVAL_SUBSET_SIZE, seed=EVAL_SUBSET_SEED):

    processed_dir = Path(processed_dir)
    test_rows = load_jsonl(processed_dir / dataset_name / f'{dataset_name}_test.jsonl')
    ids_fp = processed_dir / 'eval_subsets' / f'{dataset_name}_test_eval_ids.json'
    if ids_fp.exists():
        with open(ids_fp) as f:
            ids = json.load(f)
    else:
        all_ids = [r['id'] for r in test_rows]
        if len(all_ids) <= n:
            ids = all_ids 
        else:
            rng = random.Random(seed)
            ids = sorted(rng.sample(all_ids, n))
        ids_fp.parent.mkdir(parents=True, exist_ok=True)
        with open(ids_fp, 'w') as f:
            json.dump(ids, f, indent=2)
        print(f'[eval subset] sampled {len(ids)} test ids for {dataset_name} -> {ids_fp}')
    by_id = {r['id']: r for r in test_rows}
    return [by_id[i] for i in ids]


def build_label_maps(label_list):
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label


def tokenize_and_align(rows, tokenizer, label2id, max_length=256):
    from datasets import Dataset

    def encode(batch):
        enc = tokenizer(batch['tokens'], is_split_into_words=True,
                        truncation=True, max_length=max_length)
        all_labels = []
        for i, tags in enumerate(batch['tags']):
            word_ids = enc.word_ids(batch_index=i)
            labels, prev = [], None
            for wid in word_ids:
                if wid is None:
                    labels.append(-100)
                elif wid != prev:
                    labels.append(label2id[tags[wid]])
                else:
                    labels.append(-100)
                prev = wid
            all_labels.append(labels)
        out = dict(enc)
        out['labels'] = all_labels
        return out

    ds = Dataset.from_list([{'tokens': r['tokens'], 'tags': r['tags']} for r in rows])
    return ds.map(encode, batched=True, remove_columns=['tokens', 'tags'])


def predict_tags(model, tokenizer, rows, batch_size=32, max_length=256, device=None):
    import torch
    if device is None:
        device = pick_device()
    model = model.to(device).eval()
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    out = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        enc = tokenizer([r['tokens'] for r in batch], is_split_into_words=True,
                        truncation=True, max_length=max_length,
                        padding=True, return_tensors='pt')
        with torch.no_grad():
            logits = model(**{k: v.to(device) for k, v in enc.items()}).logits.cpu()
        pred_ids = logits.argmax(-1)
        for i, r in enumerate(batch):
            word_ids = enc.word_ids(batch_index=i)
            tags, prev = [], None
            for pos, wid in enumerate(word_ids):
                if wid is not None and wid != prev:
                    tags.append(id2label[int(pred_ids[i][pos])])
                prev = wid
     
            while len(tags) < len(r['tokens']):
                tags.append('O')
            out.append(tags)
    return out


In [ ]:
import json
import shutil
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, '.')  
import ner_common_utils as ncu

ncu.mount_drive_if_colab()
processed_dir = ncu.resolve_processed_dir()
OUTPUT_ROOT = ncu.resolve_output_root(processed_dir)

utils_dir = OUTPUT_ROOT / 'utils'
utils_dir.mkdir(parents=True, exist_ok=True)
shutil.copy('ner_common_utils.py', utils_dir / 'ner_common_utils.py')

print('processed dir :', processed_dir)
print('output root   :', OUTPUT_ROOT)
print('utils copy    :', utils_dir / 'ner_common_utils.py')

In [ ]:
SMOKE_TEST = False

MODEL_NAME = 'distilbert-base-cased'

LEARNING_RATE = 5e-5
BATCH_SIZE = 16
EPOCHS = 3 if not SMOKE_TEST else 1
MAX_LENGTH = 256

suffix = '_smoke' if SMOKE_TEST else ''
model_out_dir = OUTPUT_ROOT / 'models' / f'baseline_conll{suffix}'
results_dir = OUTPUT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
print('SMOKE_TEST:', SMOKE_TEST, '| checkpoint ->', model_out_dir)

In [ ]:
# load the processed CoNLL-2003 splits
conll = {}
for split in ['train', 'validation', 'test']:
    conll[split] = ncu.load_jsonl(processed_dir / 'conll2003' / f'conll2003_{split}.jsonl')
    print(split, len(conll[split]), 'sentences')
label_list = sorted({t for rows in conll.values() for r in rows for t in r['tags']})
label2id, id2label = ncu.build_label_maps(label_list)
print(len(label_list), 'labels:', label_list)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_rows = conll['train']
if SMOKE_TEST:
    train_rows = train_rows[:300]  # just enough to see the loss move

train_ds = ncu.tokenize_and_align(train_rows, tokenizer, label2id, max_length=MAX_LENGTH)
print(train_ds)

In [ ]:
import torch
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)

ncu.set_seed(42)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / 'tmp_trainer' / f'baseline{suffix}'),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_strategy='epoch',
    save_strategy='no',      
    report_to='none',
    disable_tqdm=True,       
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    processing_class=tokenizer,
)

t0 = time.time()
trainer.train()
train_seconds = time.time() - t0
print(f'training took {train_seconds:.0f}s on {ncu.pick_device()}')

model_out_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_out_dir)
tokenizer.save_pretrained(model_out_dir)
print('saved checkpoint ->', model_out_dir)

In [ ]:
in_domain = {}
for split in ['validation', 'test']:
    rows = conll[split]
    if SMOKE_TEST:
        rows = rows[:100]
    preds = ncu.predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH)
    in_domain[split] = ncu.compute_entity_f1([r['tags'] for r in rows], preds)
    m = in_domain[split]
    print(f"conll2003 {split}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} "
          f"({m['support']} gold entities)")

pd.DataFrame(in_domain['test']['per_type']).T

## Decision 2 - the cross-domain "degradation" check and the label-set mismatch

The label harmonization cell in `01_eda_ner.ipynb` already showed the problem: the exact-string
intersection of entity types across the three datasets is **empty** (CoNLL: LOC/MISC/ORG/PER;
WNUT-17: person/location/corporation/group/creative-work/product; SciERC:
Generic/Material/Method/Metric/OtherScientificTerm/Task). So the CoNLL head's output classes
don't literally correspond to any target-domain class, and pointing the baseline at WNUT/SciERC
test data "as is" would produce a meaningless type-level score.

The two target domains aren't symmetric, though. WNUT-17 shares real concepts with CoNLL even
though the strings differ (person~PER, location~LOC, corporation/group~ORG). SciERC shares
nothing -- Method/Task/Material are scientific-writing concepts with no real-world-entity
analogue, so no hand mapping can fix that side.

**What we do here** (a combination of options A and C that we considered):

1. **Both domains, one common yardstick:** collapse gold and predicted tags to a single
   "part of any entity vs. not" pseudo-type and score span-level detection F1. This measures
   how well the CoNLL baseline still *finds* entity mentions in the new domain, with type
   semantics removed. We compute the same binary score on CoNLL's own test set as the
   in-domain reference, so "degradation" is a like-for-like drop on the same scale.
2. **WNUT-17 only, an extra type-level view:** map gold person->PER, location->LOC,
   corporation->ORG, group->ORG; drop gold creative-work/product spans (no CoNLL analogue)
   and drop predicted MISC spans (no WNUT analogue). Caveat we accept: predictions that
   overlap a dropped gold span count as false positives, so measured precision is slightly
   pessimistic. Good enough for a rough degradation check.
3. **SciERC gets no type mapping** -- a forced one would produce a number that means nothing.

Scope note: this decision only affects the degradation check in this notebook. Arms 1/2/3
always train a fresh head on each target dataset's own native labels, so the main
comparison is untouched either way.

All target-domain predictions below run on the **fixed test subsamples** (n<=200 per dataset,
sampled once with seed 42, ids saved under `data/processed/eval_subsets/`). Every later
notebook evaluates on these exact sentences -- the LLM arm can't afford full test sets, and
mixing eval sets across methods would invalidate the comparison.

In [ ]:
eval_subsets = {}
for ds in ncu.TARGET_DATASETS:
    eval_subsets[ds] = ncu.make_or_load_eval_subset(processed_dir, ds)
    print(f'{ds}: fixed eval subset of {len(eval_subsets[ds])} test sentences')

In [ ]:
from seqeval.metrics.sequence_labeling import get_entities


def rebuild_bio(length, spans):
    tags = ['O'] * length
    for etype, s, e in spans:
        tags[s] = f'B-{etype}'
        for k in range(s + 1, e + 1):
            tags[k] = f'I-{etype}'
    return tags


def collapse_to_binary(tags):
    
    return rebuild_bio(len(tags), [('ENT', s, e) for _, s, e in get_entities(tags)])


WNUT_TO_CONLL = {
    'person': 'PER',
    'location': 'LOC',
    'corporation': 'ORG',
    'group': 'ORG',
}
CONLL_KEEP = {'PER': 'PER', 'LOC': 'LOC', 'ORG': 'ORG'}  # MISC intentionally dropped


def map_types(tags, mapping):
    kept = [(mapping[t], s, e) for t, s, e in get_entities(tags) if t in mapping]
    return rebuild_bio(len(tags), kept)

In [ ]:
cross_domain = {'binary_detection': {}, 'wnut17_mapped_types': None}

# in-domain reference on the same binary yardstick
conll_test_rows = conll['test'] if not SMOKE_TEST else conll['test'][:100]
conll_test_preds = ncu.predict_tags(model, tokenizer, conll_test_rows, max_length=MAX_LENGTH)
cross_domain['binary_detection']['conll2003_test_reference'] = ncu.compute_entity_f1(
    [collapse_to_binary(r['tags']) for r in conll_test_rows],
    [collapse_to_binary(p) for p in conll_test_preds])

target_preds = {}
for ds in ncu.TARGET_DATASETS:
    rows = eval_subsets[ds] if not SMOKE_TEST else eval_subsets[ds][:40]
    preds = ncu.predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH)
    target_preds[ds] = (rows, preds)
    cross_domain['binary_detection'][ds] = ncu.compute_entity_f1(
        [collapse_to_binary(r['tags']) for r in rows],
        [collapse_to_binary(p) for p in preds])

# WNUT-17 extra: type-level score through the hand mapping
rows, preds = target_preds['wnut17']
cross_domain['wnut17_mapped_types'] = ncu.compute_entity_f1(
    [map_types(r['tags'], WNUT_TO_CONLL) for r in rows],
    [map_types(p, CONLL_KEEP) for p in preds])

summary_rows = []
for name, m in cross_domain['binary_detection'].items():
    summary_rows.append({'eval_on': name, 'measure': 'binary detection',
                         'precision': m['precision'], 'recall': m['recall'],
                         'f1': m['f1'], 'gold_spans': m['support']})
m = cross_domain['wnut17_mapped_types']
summary_rows.append({'eval_on': 'wnut17', 'measure': 'mapped types (PER/LOC/ORG)',
                     'precision': m['precision'], 'recall': m['recall'],
                     'f1': m['f1'], 'gold_spans': m['support']})
pd.DataFrame(summary_rows).round(3)

In [ ]:
baseline_results = {
    'model_name': MODEL_NAME,
    'checkpoint_dir': str(model_out_dir),
    'smoke_test': SMOKE_TEST,
    'train_sentences': len(train_rows),
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'train_seconds': train_seconds,
    'device': ncu.pick_device(),
    'in_domain': in_domain,
    'cross_domain': cross_domain,
    'eval_subset_sizes': {ds: len(eval_subsets[ds]) for ds in ncu.TARGET_DATASETS},
    'wnut_type_mapping': WNUT_TO_CONLL,
}

out_fp = results_dir / f'baseline_results{suffix}.json'
with open(out_fp, 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('saved ->', out_fp)